<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-4-fine-tuning/Module_4_Session_1_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 — Session 1: Dataset Preparation
## Goal: Build a high-quality JSONL training dataset for Swiggy customer support fine-tuning

In [1]:
# What one fine-tuning example looks like
# This is the Alpaca format — used by Unsloth, HuggingFace, and most fine-tuning frameworks

example = {
    "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
    "input": "My order #SW78234 was supposed to arrive 45 minutes ago. Where is it?",
    "output": "I completely understand how frustrating a delayed order can be! I've checked order #SW78234 and can see it's currently 10 minutes away. The delivery partner faced unexpected traffic near Koramangala. As a goodwill gesture, I'm adding ₹50 to your Swiggy wallet. Your order will arrive shortly — thank you for your patience!"
}

# Print it nicely
import json
print(json.dumps(example, indent=2))

{
  "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
  "input": "My order #SW78234 was supposed to arrive 45 minutes ago. Where is it?",
  "output": "I completely understand how frustrating a delayed order can be! I've checked order #SW78234 and can see it's currently 10 minutes away. The delivery partner faced unexpected traffic near Koramangala. As a goodwill gesture, I'm adding \u20b950 to your Swiggy wallet. Your order will arrive shortly \u2014 thank you for your patience!"
}


In [2]:
# These are the 6 types of Swiggy complaints we want our model to handle
# More categories = more diverse and robust model

categories = [
    "order_delay",        # Order is late
    "wrong_item",         # Wrong food delivered
    "refund_request",     # Customer wants money back
    "missing_item",       # Item missing from order
    "app_issue",          # Promo code / payment not working
    "delivery_behavior"   # Rude delivery partner
]

print(f"Total categories: {len(categories)}")
print("We will generate 5 examples per category = 30 total examples")

Total categories: 6
We will generate 5 examples per category = 30 total examples


In [3]:
# Seed examples — written by us, high quality, one per category
# These set the standard for everything the LLM will generate after

seed_examples = [
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "It's been over an hour and my order hasn't arrived. I'm very hungry!",
        "output": "I sincerely apologize for this delay! I've tracked your order and see the delivery partner is 8 minutes away due to heavy rain in your area. I've added ₹75 to your Swiggy wallet for this inconvenience. We value your patience and hope to serve you better next time!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "I ordered a Veg Burger but received a Chicken Burger. I am vegetarian!",
        "output": "I'm truly sorry for this serious mistake! I've immediately raised a complaint with the restaurant and initiated a full refund of ₹249 to your original payment method within 5-7 business days. I've also added a ₹100 coupon for your next order. We take dietary requirements very seriously."
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order was cancelled automatically but money was deducted from my account.",
        "output": "I apologize for this inconvenience! I can see the order was cancelled due to a restaurant issue. Your payment of ₹380 will be refunded within 5-7 business days to your original account. To compensate for the trouble, I've added ₹50 to your Swiggy wallet. Please feel free to reorder!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order arrived but the french fries were missing. I paid ₹80 extra for them!",
        "output": "I'm sorry about the missing fries! I've verified your order and confirmed the fries were not delivered. I've refunded ₹80 to your Swiggy wallet immediately. Thank you for flagging this — it helps us improve!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "The delivery partner was very rude to me and threw the food bag at my door.",
        "output": "I sincerely apologize for this completely unacceptable behavior! I've flagged this incident for immediate review and the delivery partner will face strict action. I've added ₹150 to your Swiggy wallet as compensation. You deserve respectful service always — thank you for reporting this."
    }
]

print(f"Seed examples created: {len(seed_examples)}")

Seed examples created: 5


In [5]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.1 MB/s eta 0:00:00


In [6]:
import os
from groq import Groq

# Load API key from Colab Secrets
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq()

print("Groq client ready!")

Groq client ready!


In [7]:
def generate_examples_for_category(category, n=4):
    """
    Use Groq LLM to generate n synthetic training examples for a given category.

    Input: category name (string), number of examples to generate
    Output: list of dictionaries in Alpaca format
    """

    prompt = f"""You are creating training data for a Swiggy customer support AI.

Generate exactly {n} different customer support examples for the category: {category}

Rules:
- instruction must always be: "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution."
- input must be a realistic Indian customer complaint (1-2 sentences)
- output must be an empathetic agent response (3-4 sentences, always offer a solution)
- Use Indian context: mention ₹ amounts, Indian cities like Mumbai/Delhi/Bengaluru/Chennai

Return ONLY a valid JSON array, no explanation, no markdown, no extra text.
Format:
[
  {{"instruction": "...", "input": "...", "output": "..."}},
  {{"instruction": "...", "input": "...", "output": "..."}}
]"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8
    )

    raw_text = response.choices[0].message.content

    # json.loads converts the JSON string into a Python list
    examples = json.loads(raw_text)

    return examples

# Test with just one category first
print("Testing with category: order_delay")
test_examples = generate_examples_for_category("order_delay", n=2)
print(f"Generated: {len(test_examples)} examples")
print(json.dumps(test_examples[0], indent=2))

Testing with category: order_delay
Generated: 2 examples
{
  "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
  "input": "I've been waiting for my \u20b9250 Swiggy order from Delhi to be delivered since 2 hours, it's really frustrating.",
  "output": "Sorry to hear that your order is delayed, Mr./Ms. [Customer Name]. We understand how frustrating it can be to wait for your food. I'd be happy to help resolve this issue. Could you please provide your order id so I can check on the status and expedite the delivery for you?"
}


In [8]:
# Generate 4 examples per category × 6 categories = 24 synthetic + 5 seed = 29 total

all_examples = seed_examples.copy()  # Start with our 5 hand-written seed examples

for category in categories:
    print(f"Generating examples for: {category}...")

    try:
        new_examples = generate_examples_for_category(category, n=4)
        all_examples.extend(new_examples)  # adds all items from new_examples into all_examples
        print(f"  Added {len(new_examples)} examples. Total so far: {len(all_examples)}")

    except Exception as e:
        print(f"  Error for {category}: {e}")

print(f"\nFinal total: {len(all_examples)} examples")

Generating examples for: order_delay...
  Added 4 examples. Total so far: 9
Generating examples for: wrong_item...
  Added 4 examples. Total so far: 13
Generating examples for: refund_request...
  Error for refund_request: Expecting ',' delimiter: line 20 column 258 (char 2304)
Generating examples for: missing_item...
  Added 4 examples. Total so far: 17
Generating examples for: app_issue...
  Added 4 examples. Total so far: 21
Generating examples for: delivery_behavior...
  Added 4 examples. Total so far: 25

Final total: 25 examples


In [9]:
def validate_examples(examples):
    """
    Check every example has all 3 required fields and output is not too short.

    Input: list of example dictionaries
    Output: list of valid examples only
    """
    required_fields = ["instruction", "input", "output"]
    valid = []
    issues = 0

    for i, ex in enumerate(examples):
        # Check all 3 fields exist
        if not all(field in ex for field in required_fields):
            print(f"Example {i}: Missing field — skipping")
            issues += 1
            continue

        # Output less than 20 words is too short — likely a bad example
        output_words = len(ex["output"].split())
        if output_words < 20:
            print(f"Example {i}: Output too short ({output_words} words) — skipping")
            issues += 1
            continue

        valid.append(ex)

    print(f"\nValidation complete:")
    print(f"  Total checked : {len(examples)}")
    print(f"  Valid examples: {len(valid)}")
    print(f"  Issues found  : {issues}")

    return valid

validated_examples = validate_examples(all_examples)


Validation complete:
  Total checked : 25
  Valid examples: 25
  Issues found  : 0


In [10]:
import random

# shuffle so examples are not grouped by category
random.seed(42)  # seed=42 means same shuffle every time you run — reproducible
random.shuffle(validated_examples)

# 80/20 split
split_point = int(len(validated_examples) * 0.8)
train_data = validated_examples[:split_point]
test_data  = validated_examples[split_point:]

print(f"Total examples : {len(validated_examples)}")
print(f"Train examples : {len(train_data)}")
print(f"Test examples  : {len(test_data)}")

Total examples : 25
Train examples : 20
Test examples  : 5


In [11]:
def save_jsonl(data, filepath):
    """
    Save a list of dictionaries as a JSONL file.
    JSONL = one JSON object per line (not a JSON array)

    Input: list of dicts, file path string
    Output: saves file to disk
    """
    with open(filepath, "w") as f:
        for item in data:
            # json.dumps converts one dict to a JSON string
            # "\n" adds a newline after each example
            f.write(json.dumps(item) + "\n")

    print(f"Saved {len(data)} examples to {filepath}")

# Save both files
save_jsonl(train_data, "swiggy_train.jsonl")
save_jsonl(test_data,  "swiggy_test.jsonl")

Saved 20 examples to swiggy_train.jsonl
Saved 5 examples to swiggy_test.jsonl


In [12]:
import pandas as pd

def load_jsonl(filepath):
    """
    Load a JSONL file back into a list of dictionaries.

    Input: file path string
    Output: list of dicts
    """
    data = []
    with open(filepath, "r") as f:
        for line in f:
            # .strip() removes trailing whitespace and newline characters
            data.append(json.loads(line.strip()))
    return data

# Load train file back
train_loaded = load_jsonl("swiggy_train.jsonl")

# Convert to DataFrame for easy viewing
df = pd.DataFrame(train_loaded)

# Add word count columns so we can check quality
df["input_length"]  = df["input"].apply(lambda x: len(x.split()))
df["output_length"] = df["output"].apply(lambda x: len(x.split()))

print(f"Total rows loaded: {len(df)}")
print(f"Average input length  : {df['input_length'].mean():.1f} words")
print(f"Average output length : {df['output_length'].mean():.1f} words")
print("\nPreview:")
print(df[["input", "output_length"]].head(5))

Total rows loaded: 20
Average input length  : 25.4 words
Average output length : 55.0 words

Preview:
                                               input  output_length
0  I received my order of ₹300 worth of food from...             68
1  I ordered a custom pizza for ₹250 from a popul...             61
2  I ordered a veg biryani for ₹120 from Swiggy i...             54
3  I placed an order on Swiggy in Bengaluru and i...             62
4  I've been charged ₹1200 for a delivery fee on ...             59
